A few different approaches to automatically identifying sections of the interviews. None was massively successful but they did make manually labelling the sections slightly quicker.

In [ ]:
import pandas as pd
import random

# import numpy as np
import pandas as pd
import torch

from sentence_transformers import SentenceTransformer, util

from dsp_interview_transcripts import PROJECT_DIR

# Set random seeds
RANDOM_SEED = 42
from numpy import random as npr

npr.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

In [21]:
combined_transcripts = pd.read_csv(f"{PROJECT_DIR}/data/bit_france/converted/combined_transcripts_with_roles.csv")

# Approach 1: semantic similarity

In [3]:
phrases = [
    "je vais vous montrer différentes maquettes",
    "je vais vous partager du coup des maquettes",
    "je vais vous montrer maintenant des outils",
    "je vais vous partager mon écran",
    "maquettes et interventions",
    "je vais vous présenter des outils",
    "on va passer à une deuxième partie"
]

In [4]:
import re

def split_into_chunks(text):
    return [chunk.strip() for chunk in re.split(r'[.,;]', text) if chunk.strip()]

In [ ]:
df_interviewer = combined_transcripts[combined_transcripts['role'] == 'other']

model_name = "dangvantuan/sentence-camembert-large"
SENTENCE_MODEL = SentenceTransformer(model_name)

In [ ]:
df_interviewer['chunks'] = df_interviewer['text'].apply(split_into_chunks)
df_interviewer_long = df_interviewer.explode('chunks')
df_interviewer_long

In [ ]:
text_embeddings = SENTENCE_MODEL.encode(df_interviewer_long['chunks'].tolist(), convert_to_tensor=True)
phrase_embeddings = SENTENCE_MODEL.encode(phrases, convert_to_tensor=True)

In [8]:
cosine_similarities = util.cos_sim(text_embeddings, phrase_embeddings)

In [9]:
most_similar_indices = cosine_similarities.argmax(dim=1)  # Get the index of the most similar phrase for each text
most_similar_scores = cosine_similarities.max(dim=1).values  # Get the similarity scores

# Add similarity scores and the most similar phrase to the dataframe
df_interviewer_long['most_similar_phrase'] = [phrases[idx] for idx in most_similar_indices]
df_interviewer_long['similarity_score'] = most_similar_scores.tolist()

In [ ]:
df_interviewer_long.head()

In [ ]:
df_interviewer_long['similarity_score'].hist()

In [ ]:
def select_rows(group, threshold=0.8):
    # Filter rows with similarity_score >= 0.5
    filtered = group[group['similarity_score'] >= threshold]
    if not filtered.empty:
        return filtered
    else:
        # If no rows meet the threshold, return the row with the max similarity_score
        return group.loc[group['similarity_score'].idxmax()]
    
max_similarity_rows = df_interviewer_long.groupby('file_name', group_keys=False).apply(select_rows)
max_similarity_rows

In [ ]:
# max_similarity_rows = df_interviewer.loc[df_interviewer.groupby('file_name')['similarity_score'].idxmax()]
max_similarity_rows['file_name'].value_counts()

In [ ]:
max_similarity_rows['new_section'] = 'Yes'

out_data = pd.merge(combined_transcripts, max_similarity_rows[['file_name', 'text', 'chunks','most_similar_phrase','similarity_score','new_section']], on=['file_name', 'text'], how='left')

out_data.head()

In [15]:
out_data.to_csv('data_labelled_with_new_sections.csv', index=False)

# Approach 2: regex matches

In [27]:
combined_transcripts = pd.read_csv(f"{PROJECT_DIR}/data/bit_france/converted/combined_transcripts_with_roles.csv")

In [28]:
phrases_for_exact_matching = ["partager mon écran", "deuxième partie", "maquettes"]

In [29]:
for phrase in phrases_for_exact_matching:
    column_name = phrase.replace(" ", "_")  # Replace spaces with underscores for column names
    combined_transcripts[column_name] = combined_transcripts.apply(
        lambda row: phrase in row['text'] and row['role'] == 'other',
        axis=1
    )

In [ ]:
len(combined_transcripts[combined_transcripts["maquettes"]==1])

In [ ]:
combined_transcripts.head()

In [33]:
combined_transcripts.to_csv('ALT_data_labelled_with_new_sections.csv', index=False)

# Approach 3

In [3]:
import pandas as pd

from langchain.prompts import PromptTemplate
from langchain_community.chat_models import ChatOllama
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel
from pydantic import Field

from dsp_interview_transcripts import logger, PROJECT_DIR

In [4]:
combined_transcripts = pd.read_csv(f"{PROJECT_DIR}/data/bit_france/converted/combined_transcripts_with_roles.csv")

In [ ]:
texts = combined_transcripts[(combined_transcripts['file_name']=='C4 - Salariés - E65 - Linda_2011.txt') & (combined_transcripts['role']=='other')]
len(texts)

In [15]:
class NewSection(BaseModel):
    """Model for identifying where interview changes"""

    out_text: str = Field(description="The exact text from the input where the interviewer begins a new section")

prompt = (
        """
        Here is a list of excerpts from an interview in French:\n\n
        ```
        {texts}\n\n
        ```
        
        Your job is to identify the excerpt where the interviewer moves to a new section of the interview.
        In the first section of the interview, the interviewer asks questions about the interviewee's work life.
        In the second section, the interviewer will share screen and show the interviewee different interventions or tools.
        I need you to identify the exact text where the interviewer moves to this second section of the interview.
        The interviewer may say something like "je vais vous montrer différentes maquettes", or "je vais vous partager mon écran",
        or "on va passer à une deuxième partie".
        
        Based on the information above, please provide the **exact text** where the interviewer starts a new section of the interview as a JSON dict with the following field:
        - out_text: The exact text from the input where the interviewer starts a new section of the interview
    \n
    Provide nothing except for this JSON dict.
    """
    )

parser = JsonOutputParser(pydantic_object=NewSection)

final_prompt = PromptTemplate(
    template=prompt,
    input_variables=["texts"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

model = "llama3.2"

ollama_model = ChatOllama(model=model, temperature=0)

llm_chain = final_prompt | ollama_model | parser

In [16]:
def find_text(
    df: pd.DataFrame,
    llm_chain,
    text_col: str = "text",
):
    
    files = df['file_name'].unique().tolist()

    results = {}

    for file in files:
        logger.info(f"Processing file {file}")
        temp_df = df[(df['file_name'] == file) & (df['role']=='other')]
        docs = temp_df[text_col].values[0]
        # logger.info(f"Texts: {docs}")

        try:
            output = llm_chain.invoke({"texts": docs})
            logger.info(f"Key text: {output['out_text']}")
            results[file] = output

        except Exception as e:
            logger.error(f"Error processing file {file}: {str(e)}")
            results[file] = {"error": str(e)}

    return results

In [ ]:
results = find_text(
        combined_transcripts, llm_chain, text_col="text"
    )

In [ ]:
results

# Check data

In [30]:
labelled_data = pd.read_csv("bit_france_interview_sections - ALT_data_labelled_with_new_sections.csv")

In [ ]:
labelled_data.head()

In [ ]:
result = labelled_data[labelled_data['new_section'] == '1'].groupby('file_name').size()
result = result.reset_index(name='count_new_section')
result

In [34]:
missing_files = []
for file in labelled_data['file_name'].unique():
    if file not in result['file_name'].values:
        missing_files.append(file)

In [ ]:
missing_files

In [ ]:
labelled_data.head()

In [ ]:
labelled_data['profession'].value_counts()

In [ ]:
labelled_data.columns

In [ ]:
filtered_data = labelled_data[['speaker_id', 'text', 'file_name', 'profession', 'informant', 'role', 'new_section']].query('profession != "Médecins du travail"')
len(labelled_data) - len(filtered_data)

In [ ]:
def filter_after_new_section(group):
    # Find the index of the first occurrence of `new_section == 1`
    new_section_idx = group[group['new_section'] == '1'].index
    if not new_section_idx.empty:
        # Keep rows after the first occurrence
        return group.loc[new_section_idx[0] + 1 :]
    return pd.DataFrame()  # Return an empty DataFrame if no `new_section == 1` is found

maquettes_df = filtered_data.groupby('file_name', group_keys=False).apply(filter_after_new_section)
len(filtered_data) - len(maquettes_df)


In [ ]:
len(maquettes_df['file_name'].unique()) - len(filtered_data['file_name'].unique())